# Dental Caries AI — dataset inventory and real-model check (Colab)

Runs against the datasets and trained weights **in your Google Drive**; nothing is uploaded to GitHub.
1. Inventory the datasets  2. Check model metadata  3. Load the real weights and run real inference  4. Run the test suite.

Set `DATA_ROOT` to the Drive folder that holds `caries_FDI_dataset_final/`, `icdascl/`, `caries/`, `healthy/` and `pipeline_stage1_to_4_corrected/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/dentalcaries'   # <- edit (e.g. a 'Shared with me' shortcut path)
MODELS_DIR = f'{DATA_ROOT}/pipeline_stage1_to_4_corrected'
import os; os.environ['DCAI_DATA_ROOT'] = DATA_ROOT; os.environ['DCAI_MODELS_DIR'] = MODELS_DIR
print(os.listdir(DATA_ROOT))

In [ ]:
!git clone -q https://github.com/Sandy24242/seminar-work.git /content/seminar-work || git -C /content/seminar-work pull -q
%cd /content/seminar-work
# Colab already has TensorFlow/Keras; keep them (the models were saved with Colab's versions).
!pip -q install fastapi 'uvicorn[standard]' python-multipart httpx pytest pypdf reportlab streamlit

## 1. Dataset inventory
Image counts, folder tree, labels and their source, YOLO problems, corrupt files, duplicates (dHash), views (reported separately — views are **not** labels). `--check-zips` CRC-checks every archive member (slow).

In [ ]:
!python ml/inventory_datasets.py --out /content/dataset_inventory --dup-hamming 0 --check-zips
from IPython.display import Markdown; Markdown(open('/content/dataset_inventory.md').read())

## 2. Metadata vs website contract (no weights needed)

In [ ]:
!python ml/check_metadata.py "$MODELS_DIR"

## 3. Real weights: load + real inference
Uses the trained `.keras` files only (the script refuses test stand-ins). Point `--images` at held-out photos, e.g. a few files from the test split.

In [ ]:
!cd backend && python scripts/check_models.py
TEST_IMAGES = '/content/seminar-work/backend/assets/clinical_caries_example.jpg'   # <- edit: real held-out photos
!cd backend && python scripts/smoke_test_real_models.py --models "$MODELS_DIR" --images "$TEST_IMAGES"

## 4. Notebook parity with the real weights + full test suite
The parity test compares the website with the training notebook's own code on the **real** models. The rest of the suite uses test stand-ins (code paths only).

In [ ]:
os.environ['DCAI_PARITY_MODELS_DIR'] = MODELS_DIR
os.environ['DCAI_PARITY_IMAGES'] = os.path.dirname(TEST_IMAGES)
!cd backend && python -m pytest -q tests/test_notebook_parity.py
!cd backend && python -m pytest -q
!python -m pytest -q ml/tests

## Training
Train with `notebooks/Dental_Caries_Pipeline_Stage1_to_Stage4_with_FDI_Stage2.ipynb` on a GPU runtime (see `ml/README.md`). It writes new `.keras` + metadata into the output folder; re-run sections 2–4 afterwards.